# 🧪 Data Quality Checks

Formalizes the DQ story: today, Silver's Cell 8 ("SILVER DATA QUALITY
REPORT") only `print()`s null counts and disappears after the run — there's
no record of quality over time. This notebook adds the two pillars Module
5.6's DQ framework has that weren't covered yet:

1. **Input Validation** — the same null/domain checks Silver's Cell 8
   already runs, reused here, but *persisted* to a table instead of printed.
2. **Transformation Checks** — row-count reconciliation across
   Bronze→Silver, something the pipeline has never measured before.
3. **Output Verification** — sanity checks on Gold (non-empty, no duplicate
   grain, key columns populated).

**Output:** `rankrangers_project_data.dq.dq_metrics` — one row per
`(run_id, rule_name)`, **appended** every run (unlike Silver/Gold's
`overwrite` convention) so quality is trackable run-over-run.

**Run this after `bronze_delta_ingest.ipynb` → `bronze_to_silver_v2.ipynb`
→ `silver_to_gold_v2.ipynb`.**

## Cell 1 — Config

In [ ]:
import uuid
from datetime import datetime, timezone

CATALOG      = 'rankrangers_project_data'
BRONZE_TABLE = f'{CATALOG}.bronze.mhcet_allotments_raw'
SILVER_TABLE = f'{CATALOG}.silver.mhcet_allotments'
GOLD_TABLE   = f'{CATALOG}.gold.mhcet_cutoffs'
DQ_TABLE     = f'{CATALOG}.dq.dq_metrics'

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.dq')

RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.now(timezone.utc)
print(f'Run ID: {RUN_ID}')
print(f'Run TS: {RUN_TS.isoformat()}')

# Accumulate rule results here, write them all to dq_metrics in one shot
# at the end (Cell 6).
results = []

def log_rule(layer, rule_name, rows_checked, rows_failed, warn_pct, fail_pct, notes=''):
    pct_failed = round(100.0 * rows_failed / rows_checked, 2) if rows_checked else 0.0
    if pct_failed > fail_pct:
        status = 'FAIL'
    elif pct_failed > warn_pct:
        status = 'WARN'
    else:
        status = 'PASS'
    icon = {'PASS': '✅', 'WARN': '⚠️ ', 'FAIL': '❌'}[status]
    print(f'{icon} [{layer}] {rule_name:35s}: {rows_failed:>8,}/{rows_checked:,} '
          f'failed ({pct_failed}%) -> {status}')
    results.append({
        'run_id': RUN_ID, 'run_ts': RUN_TS, 'layer': layer,
        'rule_name': rule_name, 'rows_checked': rows_checked,
        'rows_failed': rows_failed, 'pct_failed': pct_failed,
        'status': status, 'notes': notes,
    })

## Cell 2 — Input Validation (Silver)

Same null/domain rules as `bronze_to_silver_v2.ipynb` Cell 8, persisted here
instead of just printed. Thresholds distinguish **known, already-documented
gaps** (WARN, not FAIL) from genuine regressions:

- `branch_name`: README's own "Known Limitations" documents ~1.42% nulls
  (3 women's colleges with non-standard PDF layout) — WARN threshold set
  just above that baseline; FAIL only if it gets meaningfully worse.
- `seat_type`: **not previously documented anywhere** — found while testing
  Phase 2 locally: **26,368 rows (4.16%)** have a null `seat_type` in the
  real Silver table. This wasn't a known/accepted gap before, so it's
  tracked here as a first-class rule with its baseline recorded, rather
  than silently passing or silently failing. Flag this to the team — it's
  either a parser gap (some seat-type suffix pattern not covered by
  `bronze_to_silver_v2.ipynb` Cell 4's regex) or a genuine subset of rows
  that never had a seat type in the source PDFs; worth root-causing.

In [ ]:
from pyspark.sql import functions as F

silver_df = spark.table(SILVER_TABLE)
silver_total = silver_df.count()

# (warn_pct, fail_pct, notes) - warn_pct is where a documented/known gap
# sits today; fail_pct is the "something got meaningfully worse" line.
INPUT_RULES = {
    'mhtcet_score':    (0.0, 1.0,  ''),
    'seat_type':       (5.0, 10.0, 'Known gap as of 2026-07 - 4.16% baseline, '
                                   'root cause not yet identified; see Phase 3 notes'),
    'seat_gender':     (0.0, 1.0,  ''),
    'seat_pool':       (0.0, 1.0,  ''),
    'clean_category':  (0.0, 1.0,  ''),
    'institute_name':  (0.0, 1.0,  ''),
    'branch_name':     (2.0, 5.0,  'Known gap - 3 women\'s colleges excluded, '
                                   'non-standard PDF layout (README Known Limitations)'),
    'cap_round_num':   (0.0, 1.0,  ''),
}

string_cols = {name for name, dtype in silver_df.dtypes if dtype == 'string'}

for col_name, (warn_pct, fail_pct, notes) in INPUT_RULES.items():
    condition = F.col(col_name).isNull()
    if col_name in string_cols:
        condition = condition | (F.col(col_name) == '')
    rows_failed = silver_df.filter(condition).count()
    log_rule('silver', f'not_null.{col_name}', silver_total, rows_failed,
             warn_pct, fail_pct, notes)

## Cell 3 — Transformation Checks (row-count reconciliation)

The pillar the pipeline never measured before: does Bronze→Silver drop
roughly the amount of data we expect, or did something silently break?
Silver's known/accepted drops are VACANT seats, invalid `application_id`s,
null `mhtcet_score`, and the 3 women's-college `branch_name` nulls.

In [ ]:
bronze_total = spark.table(BRONZE_TABLE).count()
rows_dropped = bronze_total - silver_total

# Baseline observed when this rule was written: ~13.8% expected drop.
# WARN if meaningfully more data is being dropped than that; FAIL if far more
# (signals a real regression - e.g. a bad CSV re-ingest, or a filter bug).
log_rule('bronze_to_silver', 'row_count_reconciliation', bronze_total,
         rows_dropped, warn_pct=16.0, fail_pct=20.0,
         notes=(f'bronze={bronze_total:,}, silver={silver_total:,}, '
                f'dropped={rows_dropped:,} '
                f'({round(100*rows_dropped/bronze_total, 2)}%). '
                f'Expected causes: VACANT seats, invalid application_id, '
                f'null mhtcet_score, null branch_name (3 excluded colleges).'))

## Cell 4 — Output Verification (Gold)

Sanity checks on the table the Streamlit app actually queries: not empty,
no duplicate rows at the documented grain (`institute_code + branch_name +
clean_category + seat_gender + is_ews + is_tfws`, per
`silver_to_gold_v2.ipynb` Cell 3), and cutoff columns aren't universally
null.

In [ ]:
gold_df = spark.table(GOLD_TABLE)
gold_total = gold_df.count()

log_rule('gold', 'not_empty', 1, 0 if gold_total > 0 else 1, 0.0, 0.0,
         notes=f'{GOLD_TABLE} has {gold_total:,} rows')

GRAIN_COLS = ['institute_code', 'branch_name', 'clean_category',
              'seat_gender', 'is_ews', 'is_tfws']
dup_groups = (gold_df.groupBy(*GRAIN_COLS).count()
                      .filter(F.col('count') > 1))
dup_rows = dup_groups.count()
log_rule('gold', 'unique_grain', gold_total, dup_rows, 0.0, 0.0,
         notes=f'grain={GRAIN_COLS}')

cutoff_cols = ['cap1_cutoff', 'cap2_cutoff', 'cap3_cutoff', 'cap4_cutoff']
all_null_cutoffs = gold_df.filter(
    F.col('cap1_cutoff').isNull() & F.col('cap2_cutoff').isNull() &
    F.col('cap3_cutoff').isNull() & F.col('cap4_cutoff').isNull()
).count()
log_rule('gold', 'not_all_cutoffs_null', gold_total, all_null_cutoffs, 0.0, 1.0,
         notes=f'checked {cutoff_cols}')

## Cell 5 — Write `dq_metrics` (append — accumulates history across runs)

In [ ]:
from pyspark.sql.types import (StructType, StructField, StringType,
                                TimestampType, LongType, DoubleType)

dq_schema = StructType([
    StructField('run_id', StringType()),
    StructField('run_ts', TimestampType()),
    StructField('layer', StringType()),
    StructField('rule_name', StringType()),
    StructField('rows_checked', LongType()),
    StructField('rows_failed', LongType()),
    StructField('pct_failed', DoubleType()),
    StructField('status', StringType()),
    StructField('notes', StringType()),
])

results_df = spark.createDataFrame(results, schema=dq_schema)
results_df.write.format('delta').mode('append').saveAsTable(DQ_TABLE)

n_pass = sum(1 for r in results if r['status'] == 'PASS')
n_warn = sum(1 for r in results if r['status'] == 'WARN')
n_fail = sum(1 for r in results if r['status'] == 'FAIL')
print(f'✅ Logged {len(results)} rule results to {DQ_TABLE} for run {RUN_ID}')
print(f'   PASS={n_pass}  WARN={n_warn}  FAIL={n_fail}')
if n_fail:
    print('❌ One or more rules FAILED this run - investigate before trusting Gold/the app.')

## Cell 6 — Quick Verify (trend across runs)

In [ ]:
display(spark.sql(f'''
    SELECT run_ts, layer, rule_name, rows_checked, rows_failed, pct_failed, status
    FROM {DQ_TABLE}
    ORDER BY run_ts DESC, layer, rule_name
    LIMIT 50
'''))